# 🧪 TubeTalk RAG Evaluation — Ragas + Gemini

This notebook runs a standalone Ragas evaluation on the TubeTalk RAG pipeline.

**Metrics:**
- ✅ `answer_correctness` — Is the answer factually correct vs ground truth?
- ✅ `faithfulness` — Does the answer stick to the retrieved context?
- ✅ `answer_relevancy` — Does the answer directly address the question?
- ✅ `context_recall` — Did the retriever fetch all the relevant info?
- ✅ `answer_similarity` — How semantically close is the answer to ground truth?


In [ ]:
# ── STEP 1: Imports ────────────────────────────────────────────────────────
import os
import sys
from dotenv import load_dotenv
load_dotenv()

# Add the parent tubetalk directory to path so we can import local modules
sys.path.append(os.path.abspath('..'))

from datasets import Dataset

# Ragas metrics
from ragas.metrics import (
    answer_correctness,
    faithfulness,
    answer_relevancy,
    context_recall,
    answer_similarity
)
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# TubeTalk modules
from hybrid_retriever import create_retriever_from_url
from chatbot import ChatbotService
from langchain_core.messages import HumanMessage

print('✅ All imports successful!')

In [ ]:
# ── STEP 2: Configure Gemini as Ragas LLM + Embeddings Judge ───────────────
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

gemini_llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)
gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2-preview',
    google_api_key=GOOGLE_API_KEY
)

# Wrap for Ragas
ragas_llm = LangchainLLMWrapper(gemini_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(gemini_embeddings)

# Override the default OpenAI inside each metric
for metric in [answer_correctness, faithfulness, answer_relevancy, context_recall, answer_similarity]:
    metric.llm = ragas_llm

# Metrics that need embeddings for semantic similarity
answer_relevancy.embeddings = ragas_embeddings
answer_similarity.embeddings = ragas_embeddings

print('✅ Gemini configured as Ragas Judge!')

In [ ]:
# ── STEP 3: Evaluation Questions + Ground Truth ─────────────────────────────
# Sourced from your LangSmith dataset: 'TubeTalk.ai EVAL'
# Video: GenAI Map (Hindi) — https://youtu.be/WzvURhaDZqI

questions = [
    'What is the main goal of the Research Layer in the GenAI Map?',
    'How are ideas from the Research Layer converted into working models?',
    'How many layers are present in this GenAI architecture?',
    'What is the specific purpose of Context Engineering within the Builder Layer?',
    'What is the role of the Operations Layer in ensuring AI application reliability?',
]

ground_truth_answers = [
    'The Research Layer is where core AI innovation is born, focusing on developing new model architectures (like Transformers or Diffusion models) and learning algorithms.',
    'In the Foundation Layer, research ideas are implemented into code and trained on massive datasets using huge compute resources to create large-scale foundation models.',
    'Generative AI architecture is organized into 8 distinct layers.',
    'Context Engineering involves managing the external data and information passed to the model in real-time, such as switching between GitHub codebases, Jira tickets, or Slack discussions depending on the task.',
    'The Operations Layer focuses on deploying and running the software reliably. This includes packaging (Docker), infrastructure (Kubernetes), deployment strategies, and handling scaling.',
]

print(f'✅ Loaded {len(questions)} evaluation questions.')

In [ ]:
# ── STEP 4: Setup Retriever and Chatbot ─────────────────────────────────────
YOUTUBE_URL = 'https://youtu.be/WzvURhaDZqI'
THREAD_ID   = 'ragas-eval-notebook'

print('🔨 Building retriever...')
retriever_result = create_retriever_from_url(YOUTUBE_URL, doc_language='hindi', thread_id=THREAD_ID)
retriever_obj = retriever_result[0] if isinstance(retriever_result, tuple) else retriever_result

print('🤖 Building chatbot...')
service = ChatbotService(api_key=GOOGLE_API_KEY)
app = service.build_chatbot(retriever_obj)

print('✅ Retriever & Chatbot ready!')

In [ ]:
# ── STEP 5: Generate Answers + Retrieve Contexts ────────────────────────────
generated_answers   = []
retrieved_documents = []

for i, question in enumerate(questions):
    print(f'\n[{i+1}/{len(questions)}] Q: {question}')

    # 1. Get chatbot answer via LangGraph
    config = {'configurable': {'thread_id': f'{THREAD_ID}-{i}'}}
    result = app.invoke(
        {'messages': [HumanMessage(content=question)]},
        config=config
    )
    answer = result['messages'][-1].content
    generated_answers.append(answer)
    print(f'   ✅ Answer: {answer[:80]}...')

    # 2. Manually call retriever to get contexts for Ragas
    docs = retriever_obj(question)
    contexts = [doc.page_content for doc in docs]
    retrieved_documents.append(contexts)
    print(f'   📄 Retrieved {len(contexts)} chunks.')

print('\n✅ All answers and contexts generated!')

In [ ]:
# ── STEP 6: Build Ragas Dataset + Run Evaluation ────────────────────────────
data_samples = {
    'question'    : questions,
    'answer'      : generated_answers,
    'contexts'    : retrieved_documents,   # Already a list of lists of strings
    'ground_truth': ground_truth_answers
}

# Ensure contexts are list[list[str]] (Ragas requirement)
data_samples['contexts'] = [
    [ctx] if isinstance(ctx, str) else ctx
    for ctx in data_samples['contexts']
]

dataset = Dataset.from_dict(data_samples)

print('🧪 Running Ragas Evaluation with Gemini...')
metrics = [
    answer_correctness,
    faithfulness,
    answer_relevancy,
    context_recall,
    answer_similarity
]

# Note: We pass ragas_llm explicitly here as well for safety
score = evaluate(dataset, metrics=metrics, llm=ragas_llm, embeddings=ragas_embeddings)
print('\n✅ Evaluation complete!')

In [ ]:
# ── STEP 7: Display Results as a DataFrame ──────────────────────────────────
import pandas as pd

results_df = score.to_pandas()

# Show only the metric columns + question for readability
display_cols = ['question', 'answer_correctness', 'faithfulness', 'answer_relevancy', 'context_recall', 'answer_similarity']
print('\n📊 Evaluation Results:\n')
print(results_df[display_cols].to_string(index=False))

print('\n📈 Average Scores:')
print(results_df[['answer_correctness', 'faithfulness', 'answer_relevancy', 'context_recall', 'answer_similarity']].mean())